# Repo conventions (1 page)

This notebook is the single source of truth for **interfaces and conventions** used across the repo (especially `inventory/` and `queueing/`).

If something feels inconsistent between notebooks, update this first.

## 1) Time index conventions

### Inventory (discrete time)
- Time is **discrete**: `t = 0, 1, ..., T-1`.
- We sample exogenous information as $W_{t+1}$ **during** step `t`.
- Dynamics are written as:
  - $W_{t+1} \\sim \\mathrm{Exogenous}(S_t, X_t, t)$
  - $S_{t+1} = f(S_t, X_t, W_{t+1}, t)$
  - $C_t = c(S_t, X_t, W_{t+1}, t)$

### Queueing / scenarios (continuous time DES)
- Time is **continuous** (floating point).
- Horizon `T` means “simulate until simulation time reaches `T`”.

## 2) Shape conventions (state / action / exogenous)

### Inventory (vector conventions, always)
Inventory uses *vector* conventions **even when the dimension is 1**:
- `State S`: a 1-D numpy array of shape `(dS,)`
- `Action X`: a 1-D numpy array of shape `(dX,)`
- `Exogenous W`: a 1-D numpy array of shape `(dW,)`

The official interfaces are:
- `Policy.act(state: State, t: int, info: dict | None) -> Action`
- `ExogenousModel.sample(state: State, action: Action, t: int, rng) -> Exog`

Practical rules:
- Always pass `S0` as a vector (e.g. `np.array([80.0])`, not `80.0`).
- Policies must return vectors (e.g. `np.array([x])`, not a scalar).

### Queueing (routing + dispatch)
There are two common “state/action” notions in queueing:
- **DES simulator** (KPI evaluation): runs with continuous time and produces KPIs; policies are (routing decisions + optional dispatch decisions).
- **RL environment** (`RoutingOnlyEnv`): observation is a 1-D float vector produced by `RoutingObsSpec.encode(...)` and routing action is an integer index into the available next-node actions.

For the RL observation vector, the encoding is stable and concatenates:
`[class_onehot | from_node_onehot | slack | queue_lens | in_service | blocked]`.

## 3) Seeds and reproducibility

### Inventory
There are two seed concepts you’ll see:
- `seed0`: a **master seed** used to generate a deterministic list of episode seeds.
- `crn_step_seed`: a **per-step seed** injected into `info` on every step to support strict CRN.

The inventory simulator generates per-step seeds and then uses them to seed the per-step RNG used by the exogenous model.

### Queueing
Strict-CRN evaluation uses a simple pattern:
- Replication `r` uses `ep_seed = seed0 + r`.
- For each replication, *each policy* is simulated with the **same** `ep_seed`, so paired differences have reduced noise.

In the RL environment (`RoutingOnlyEnv`), exogenous randomness is keyed off a base seed plus counters (arrivals, service visits, routing decisions, etc.), so the same `seed` reproduces the same event stream under the same decision sequence.

### Fast mode
Some runners support a fast smoke mode via `SDAM_FAST=1` (or a `--fast` flag). Fast mode reduces replications and horizons; it should not change semantics beyond “do less work”.

## 4) Strict-CRN recipe (how to do fair A/B comparisons)

Strict CRN means: **all policies see the same randomness stream**, as much as the model allows.

### Inventory strict-CRN (recommended pattern)
1. Choose and fix: `(S0, T, n_episodes, seed0)`.
2. Use `evaluate_policies_crn_mc(...)` (or the canonical notebook pipeline helpers) so that per-step RNG seeds are shared across policies.
3. Keep policies in deterministic evaluation mode unless you intentionally want stochastic actions:
   - Evaluators default to `info["deterministic"] = True` when not provided.
   - Policies that *do* sample actions can optionally use `info["crn_step_seed"]` to make decision sampling reproducible under CRN.

Minimal example (conceptual):
```python
results, _ = system.evaluate_policies_crn_mc(
    policies={"A": pol_A, "B": pol_B},
    S0=S0,
    T=T,
    n_episodes=n_episodes,
    seed0=seed0,
    info={"deterministic": True, "det_mode": "argmax"},
)
```

### Queueing strict-CRN (recommended pattern)
1. Choose and fix: `(cfg, T, n_rep, seed0)`.
2. Use `evaluate_policies_crn_mc_queue(...)` with a single `baseline_name`.
3. Interpret paired deltas from the report (`candidate - baseline`) rather than comparing unpaired means.

Reminder: strict-CRN reduces Monte Carlo variance; it does not remove modeling bias or guarantee one policy is better.

## 5) Where to sanity-check
- Run the setup notebook: `lectures/00_setup.ipynb`.
- Inventory strict-CRN smoke suite: `poetry run sdam-inventory-smoke`.
- Queueing scenarios quick smoke: `poetry run sdam-scenarios --fast --only scenarios.scenario_01_mm1 --no-env`.